In [ ]:
import pandas as pd
import numpy as np
import pandas as pd
import lightgbm as lgb
import xgboost as xgb
from fe_utils import apply_basic_features, calculate_lrfm
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestRegressor
from sklearn.feature_selection import SelectFromModel

In [2]:
df = pd.read_csv('Dataset/Cleaned_data/customer_analytics_dataset.csv')
df.head()

,CustomerID,Transaction_ID,Transaction_Date,Product_SKU,Product_Description,Product_Category,Quantity,Avg_Price,Delivery_Charges,Coupon_Status,Month,Gender,Location,Tenure_Months,GST,Coupon_Code,Discount_pct,Offline_Spend,Online_Spend
0,17850,16679,2019-01-01,GGOENEBJ079499,Nest Learning Thermostat 3rd Gen-USA - Stainle...,Nest-USA,1,153.71,6.5,Used,Jan,M,Chicago,12,0.10,ELEC10,10.0,4500,2424.5
1,17850,16680,2019-01-01,GGOENEBJ079499,Nest Learning Thermostat 3rd Gen-USA - Stainle...,Nest-USA,1,153.71,6.5,Used,Jan,M,Chicago,12,0.10,ELEC10,10.0,4500,2424.5
2,17850,16681,2019-01-01,GGOEGFKQ020399,Google Laptop and Cell Phone Stickers,Office,1,2.05,6.5,Used,Jan,M,Chicago,12,0.10,OFF10,10.0,4500,2424.5
3,17850,16682,2019-01-01,GGOEGAAB010516,Google Men's 100% Cotton Short Sleeve Hero Tee...,Apparel,5,17.53,6.5,Not Used,Jan,M,Chicago,12,0.18,SALE10,10.0,4500,2424.5
4,17850,16682,2019-01-01,GGOEGBJL013999,Google Canvas Tote Natural/Navy,Bags,1,16.50,6.5,Used,Jan,M,Chicago,12,0.18,AIO10,10.0,4500,2424.5


In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 52924 entries, 0 to 52923
Data columns (total 19 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   CustomerID           52924 non-null  int64  
 1   Transaction_ID       52924 non-null  int64  
 2   Transaction_Date     52924 non-null  object 
 3   Product_SKU          52924 non-null  object 
 4   Product_Description  52924 non-null  object 
 5   Product_Category     52924 non-null  object 
 6   Quantity             52924 non-null  int64  
 7   Avg_Price            52924 non-null  float64
 8   Delivery_Charges     52924 non-null  float64
 9   Coupon_Status        52924 non-null  object 
 10  Month                52924 non-null  object 
 11  Gender               52924 non-null  object 
 12  Location             52924 non-null  object 
 13  Tenure_Months        52924 non-null  int64  
 14  GST                  52924 non-null  float64
 15  Coupon_Code          52924 non-null 

 CLV = AOV x Purchase Frequency x 1/Churn Rate

 AOV = TotalPrice/Total Transactions

 Purchase Frequeny = Total Transactions/Periods of Time

 Churn Rate = (customers leaving in period/ customers buyed in the period before) / 100

In [4]:
df = apply_basic_features(df)
display(df.info(), df.describe())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 52924 entries, 0 to 52923
Data columns (total 25 columns):
 #   Column               Non-Null Count  Dtype         
---  ------               --------------  -----         
 0   CustomerID           52924 non-null  object        
 1   Transaction_ID       52924 non-null  object        
 2   Transaction_Date     52924 non-null  datetime64[ns]
 3   Product_SKU          52924 non-null  object        
 4   Product_Description  52924 non-null  object        
 5   Product_Category     52924 non-null  object        
 6   Quantity             52924 non-null  int64         
 7   Avg_Price            52924 non-null  float64       
 8   Delivery_Charges     52924 non-null  float64       
 9   Coupon_Status        52924 non-null  object        
 10  Month                52924 non-null  object        
 11  Gender               52924 non-null  object        
 12  Location             52924 non-null  object        
 13  Tenure_Months        52924 non-

None

,Transaction_Date,Quantity,Avg_Price,Delivery_Charges,Tenure_Months,GST,Discount_pct,Offline_Spend,Online_Spend,Is_Discounted,Is_Used_Coupon,Invoice,Total_Revenue
count,52924,52924.000000,52924.000000,52924.000000,52924.000000,52924.000000,52924.000000,52924.000000,52924.000000,52924.000000,52924.000000,52924.000000,52924.000000
mean,2019-07-05 19:16:09.450532864,4.497638,52.237646,10.517630,26.127995,0.137462,0.198024,2830.914141,1893.109119,0.992442,0.338296,101.983198,78.563156
min,2019-01-01 00:00:00,1.000000,0.390000,0.000000,2.000000,0.050000,0.000000,500.000000,320.250000,0.000000,0.000000,4.603500,0.315700
25%,2019-04-12 00:00:00,1.000000,5.700000,6.000000,15.000000,0.100000,0.100000,2500.000000,1252.630000,1.000000,0.000000,20.160000,11.225340
50%,2019-07-13 00:00:00,1.000000,16.990000,6.000000,27.000000,0.180000,0.200000,3000.000000,1837.870000,1.000000,0.000000,45.636200,28.865160
75%,2019-09-27 00:00:00,2.000000,102.130000,6.500000,37.000000,0.180000,0.300000,3500.000000,2425.350000,1.000000,1.000000,137.400000,114.730000
max,2019-12-31 00:00:00,900.000000,355.740000,521.360000,50.000000,0.180000,0.300000,5000.000000,4556.930000,1.000000,1.000000,8979.275000,8972.775000
std,NaN,20.104711,64.006882,19.475613,13.478285,0.045825,0.082789,936.154247,807.014092,0.086608,0.473134,172.365729,149.875728


In [5]:
df['Transaction_Date'].dt.year.value_counts()

Transaction_Date
2019    52924
Name: count, dtype: int64

In [6]:
from feature_engineering_utils import calculate_clv

# Gọi hàm để tính toán và tự động gộp các cột (AOV, Purchase_Frequency, Churn_Rate, CLV) vào df
df = calculate_clv(df)

# Kiểm tra kết quả
display(df[['CustomerID', 'AOV', 'Purchase_Frequency', 'Churn_Rate', 'CLV']].head())

,CustomerID,AOV,Purchase_Frequency,Churn_Rate,CLV
0,17850,229.040314,14.75,0.008485,398142.873481
1,17850,229.040314,14.75,0.008485,398142.873481
2,17850,229.040314,14.75,0.008485,398142.873481
3,17850,229.040314,14.75,0.008485,398142.873481
4,17850,229.040314,14.75,0.008485,398142.873481


In [ ]:
def outlier_thresholds(dataframe, variable):
    low_limit = dataframe[variable].quantile(0.01)
    up_limit = dataframe[variable].quantile(0.99)
    return low_limit, up_limit

def replace_with_thresholds(dataframe, variable):
    low_limit, up_limit = outlier_thresholds(dataframe, variable)
    # Thay thế các giá trị nằm ngoài ngưỡng
    dataframe.loc[(dataframe[variable] < low_limit), variable] = low_limit
    dataframe.loc[(dataframe[variable] > up_limit), variable] = up_limit

replace_with_thresholds(df, "Quantity")
replace_with_thresholds(df, "Avg_Price")

In [8]:
max_date = df['Transaction_Date'].max()
min_date = df['Transaction_Date'].min()
cutoff_date = min_date + (max_date - min_date) * 0.75 
# Tập Quan sát (để tạo Features) và Tập Tương lai (để tạo Target)
df_obs  = df[df['Transaction_Date'] <= cutoff_date]
df_perf = df[df['Transaction_Date'] > cutoff_date]

In [9]:

# Feature Engineering
df_obs, lrfm_obs = calculate_lrfm(df_obs)

behavior_features = (
    df_obs.groupby('CustomerID')
    .agg(
        total_quantity=('Quantity', 'sum'),
        avg_quantity_per_txn=('Quantity', 'mean'),
        total_delivery_charges=('Delivery_Charges', 'sum'),
        avg_discount_pct=('Discount_pct', 'mean'),
        total_coupons_used=('Is_Used_Coupon', 'sum'),
        tenure_months=('Tenure_Months', 'max'),
        gender=('Gender', 'first'),
        location=('Location', 'first')
    )
    .reset_index()
)


# One-Hot Encoding cho các biến chữ
behavior_features = pd.get_dummies(
    behavior_features,
    columns=['gender', 'location'],
    drop_first=True
)

#  Hợp nhất LRFM và biến hành vi thành tập X hoàn chỉnh
# Ta dùng lrfm_obs merge với behavior_features, set index thành CustomerID để chuẩn bị map với y
X = lrfm_obs.merge(behavior_features, on='CustomerID').set_index('CustomerID')

# LRFM có sinh ra cột dạng chữ 'Heuristic_Segment' (ví dụ "Champions", "At Risk"), ta cần mã hóa nó luôn
if 'Heuristic_Segment' in X.columns:
    X = pd.get_dummies(X, columns=['Heuristic_Segment'], drop_first=True)

feature_cols = X.columns.tolist()


# Tạo tập mục tiêu - Y target
y = df_perf.groupby('CustomerID')['Total_Revenue'].sum().rename('target_clv')


# Kết hợp X và Y
train_df = X.join(y, how='left').fillna(0)
train_df['target_clv_log'] = np.log1p(train_df['target_clv'])


# --- 5. CHUẨN BỊ DỮ LIỆU HUẤN LUYỆN ---
X_model = train_df[feature_cols]
y_model = train_df['target_clv_log']

X_train, X_test, y_train, y_test = train_test_split(X_model, y_model, test_size=0.2, random_state=42)

print(f"Tổng số lượng Features lấy được: {len(feature_cols)} biến")
print(f"Các cột đưa vào mô hình: {feature_cols}")


Tổng số lượng Features lấy được: 23 biến
Các cột đưa vào mô hình: ['Length', 'Recency', 'Frequency', 'Monetary', 'L_Score', 'R_Score', 'F_Score', 'M_Score', 'LRFM_Score', 'total_quantity', 'avg_quantity_per_txn', 'total_delivery_charges', 'avg_discount_pct', 'total_coupons_used', 'tenure_months', 'gender_M', 'location_Chicago', 'location_New Jersey', 'location_New York', 'location_Washington Dc', 'Heuristic_Segment_Premium', 'Heuristic_Segment_Silver', 'Heuristic_Segment_Standard']


In [10]:
train_df = X.join(y, how='left').fillna(0)

train_df['target_clv_log'] = np.log1p(train_df['target_clv'])

X_model = train_df.drop(columns=['target_clv', 'target_clv_log'])
y_model = train_df['target_clv_log']

X_train, X_test, y_train, y_test = train_test_split(X_model, y_model, test_size=0.2, random_state=42)

In [11]:
fs_model = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)

selector = SelectFromModel(fs_model, threshold='mean')
# Cho mô hình học trên tập Train để chấm điểm biến
selector.fit(X_train, y_train)
# 4. Lấy danh sách các biến "chiến thắng" (được giữ lại)
selected_mask = selector.get_support()
selected_features = X_train.columns[selected_mask].tolist()
print(f"Số lượng biến được chọn lại: {len(selected_features)} biến")
print(f"Danh sách biến được chọn: {selected_features}\n")
# 5. Cập nhật lại tập Train và Test chỉ với những biến đã qua vòng tuyển chọn
X_train = X_train[selected_features]
X_test = X_test[selected_features]

Số lượng biến được chọn lại: 11 biến
Danh sách biến được chọn: ['Length', 'Recency', 'Frequency', 'Monetary', 'LRFM_Score', 'total_quantity', 'avg_quantity_per_txn', 'total_delivery_charges', 'avg_discount_pct', 'total_coupons_used', 'tenure_months']



In [14]:
# --- 6. KHỞI TẠO CÁC MÔ HÌNH ĐỂ SO SÁNH ---
# Để cấu hình tương đồng, ta giới hạn độ sâu (max_depth) và số lượng cây (n_estimators) tương đối đồng đều
models = {
    "LightGBM": lgb.LGBMRegressor(
        n_estimators=1000, learning_rate=0.01, max_depth=5, num_leaves=31, random_state=42, verbose=-1
    ),
    "XGBoost": xgb.XGBRegressor(
        n_estimators=1000, learning_rate=0.01, max_depth=5, random_state=42, objective='reg:squarederror'
    ),
    "Random Forest": RandomForestRegressor(
        n_estimators=300, max_depth=5, min_samples_split =10, random_state=42, n_jobs=-1
    )
}

results = []
feature_importances_dict = {}

# --- 7. HUẤN LUYỆN, DỰ ĐOÁN VÀ ĐÁNH GIÁ ---
print("--- ĐANG HUẤN LUYỆN VÀ SO SÁNH CÁC MÔ HÌNH ---")

for model_name, model in models.items():
    print(f">> Chạy mô hình {model_name}...")
    
    # Huấn luyện mô hình
    model.fit(X_train, y_train)
    
    # Dự đoán
    y_pred_log = model.predict(X_test)
    
    # Chuyển ngược từ dạng Log về số tiền thực tế để tính sai số
    y_pred_actual = np.expm1(y_pred_log)
    y_test_actual = np.expm1(y_test)
    
    # Tính toán các chỉ số
    mae = mean_absolute_error(y_test_actual, y_pred_actual)
    rmse = np.sqrt(mean_squared_error(y_test_actual, y_pred_actual))
    r2 = r2_score(y_test_actual, y_pred_actual)
    
    # Lưu kết quả
    results.append({
        "Model": model_name,
        "MAE (Sai số tiền)": round(mae, 2),
        "RMSE": round(rmse, 2),
        "R2 Score": round(r2, 4)
    })
    
    # Lưu độ quan trọng của đặc trưng (Feature Importances)
    if hasattr(model, 'feature_importances_'):
        importances = pd.Series(model.feature_importances_, index=selected_features)
        feature_importances_dict[model_name] = importances.sort_values(ascending=False)

# Chuyển list kết quả thành DataFrame để hiển thị đẹp mắt trên Notebook
results_df = pd.DataFrame(results).sort_values(by="MAE (Sai số tiền)", ascending=True)

print("\n=== KẾT QUẢ SO SÁNH 3 MÔ HÌNH (Xếp theo sai số thấp nhất) ===")
display(results_df)

# Tự động lấy đặc trưng quan trọng của mô hình tốt nhất (Top 1)
best_model_name = results_df.iloc[0]["Model"]
print(f"\n=== MỨC ĐỘ QUAN TRỌNG CỦA ĐẶC TRƯNG (Mô hình tốt nhất: {best_model_name}) ===")
print(feature_importances_dict[best_model_name].head(10)) # Hiển thị top 10 đặc trưng


--- ĐANG HUẤN LUYỆN VÀ SO SÁNH CÁC MÔ HÌNH ---
>> Chạy mô hình LightGBM...
>> Chạy mô hình XGBoost...
>> Chạy mô hình Random Forest...

=== KẾT QUẢ SO SÁNH 3 MÔ HÌNH (Xếp theo sai số thấp nhất) ===


,Model,MAE (Sai số tiền),RMSE,R2 Score
1,XGBoost,412.78,1329.92,-0.0201
2,Random Forest,413.62,1360.38,-0.0673
0,LightGBM,425.31,1356.65,-0.0615



=== MỨC ĐỘ QUAN TRỌNG CỦA ĐẶC TRƯNG (Mô hình tốt nhất: XGBoost) ===
LRFM_Score                0.260245
avg_discount_pct          0.094840
total_coupons_used        0.083945
Recency                   0.083655
avg_quantity_per_txn      0.078849
total_delivery_charges    0.077655
total_quantity            0.076617
Monetary                  0.073135
Frequency                 0.061037
tenure_months             0.058366
dtype: float32
